# Colab 环境初始化

每次打开新的 Colab session 时，运行此 Notebook 完成以下操作：

1. 挂载 Google Drive
2. 从 GitHub 拉取/更新代码到 Colab 本地
3. 用软链接把 Drive 中的大文件（数据集、预训练模型、已训练权重）映射到项目目录
4. 安装额外依赖
5. 验证环境

---

### Google Drive 目录结构（建议）

```
My Drive/
  AWM/
    dataset/
      highres/original/     ← HR 壁纸
      lowres_2x/original/   ← 2x 退化 LR
      lowres_4x/original/   ← 4x 退化 LR
      lowres_4x_simple/original/  ← 4x 纯 bicubic LR（待生成）
    pretrained_models/      ← APISR 预训练权重
    saved_models/           ← 训练输出的 checkpoint
    results/                ← 推理结果
```

In [ ]:
# ========================
# 0. 配置（按需修改）
# ========================

GITHUB_REPO   = 'https://github.com/<你的用户名>/AWM.git'   # ← 替换为你的 repo URL
BRANCH        = 'main'
DRIVE_ROOT    = '/content/drive/MyDrive/AWM'   # Drive 中存放大文件的根目录
PROJECT_DIR   = '/content/AWM'                 # Colab 本地项目路径

In [ ]:
# ========================
# 1. 挂载 Google Drive
# ========================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ========================
# 2. 拉取 / 更新 GitHub 代码
# ========================
import os

if os.path.isdir(PROJECT_DIR):
    print(f'项目目录已存在，拉取最新代码 ...')
    !cd {PROJECT_DIR} && git pull origin {BRANCH}
else:
    print(f'克隆仓库 ...')
    !git clone -b {BRANCH} {GITHUB_REPO} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f'\n当前工作目录: {os.getcwd()}')

In [ ]:
# ========================
# 3. 软链接 Drive 大文件到项目目录
# ========================

links = {
    'dataset':           f'{DRIVE_ROOT}/dataset',
    'pretrained_models': f'{DRIVE_ROOT}/pretrained_models',
    'saved_models':      f'{DRIVE_ROOT}/saved_models',
    'results':           f'{DRIVE_ROOT}/results',
}

for local_name, drive_path in links.items():
    local_path = os.path.join(PROJECT_DIR, local_name)
    if os.path.islink(local_path) or os.path.isdir(local_path):
        print(f'  跳过 (已存在): {local_name} → {os.readlink(local_path) if os.path.islink(local_path) else "真实目录"}')
        continue
    if not os.path.isdir(drive_path):
        os.makedirs(drive_path, exist_ok=True)
        print(f'  创建 Drive 目录: {drive_path}')
    os.symlink(drive_path, local_path)
    print(f'  链接: {local_name} → {drive_path}')

print('\n软链接创建完毕。')

In [ ]:
# ========================
# 4. 安装额外依赖
# ========================
# Colab 自带 torch、numpy、cv2、matplotlib
# 以下是可能需要额外安装的包

!pip install -q pyiqa tqdm

In [ ]:
# ========================
# 5. 验证环境
# ========================
import glob
import torch
import cv2

print(f'PyTorch:  {torch.__version__}')
print(f'CUDA:     {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'OpenCV:   {cv2.__version__}')

for name, drive_path in links.items():
    local_path = os.path.join(PROJECT_DIR, name)
    exists = os.path.isdir(local_path)
    count = len(os.listdir(local_path)) if exists else 0
    status = f'✓ ({count} items)' if exists else '✗ 不存在'
    print(f'  {name:20s} {status}')

hr_count = len(glob.glob('dataset/highres/original/*.[jp][pn]*g')) if os.path.isdir('dataset/highres/original') else 0
print(f'\nHR 图片数: {hr_count}')
print('\n环境就绪！' if hr_count > 0 else '\n⚠ 未检测到 HR 图片，请确认 Drive 目录是否正确。')